# Moral Compass Analysis via SVD Archetype Space

This notebook implements a targeted approach to moral analysis of characters using a pre-computed Singular Value Decomposition (SVD) of character-trait data.

## Approach:
1. **Trait Selection**: Decide on 1-3 "moral" traits (e.g., Kind, Altruistic, Honest).
2. **Moral Vector ($A_m$)**: Create a vector in the trait space where only our chosen moral traits are active.
3. **Archetype Space Projection**: Compute the "moral axis" in archetype space using $U^T \times A_m$.
4. **Character Projection**: Project characters onto this axis to find moral/immoral characters.
5. **Trait Alignment**: Find which other traits align with this moral axis.
6. **Moral Duality**: Add a second dimension (e.g., Masculine/Feminine or Mature/Juvenile) to see how morality shifts across that duality.

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set paths
data_path = 'data/plain/current/2000/'

def load_matrix(filename):
    return np.loadtxt(os.path.join(data_path, filename))

print("Loading data...")
traits_df = pd.read_csv(os.path.join(data_path, 'data_traits.tsv'), sep='\t')
characters_df = pd.read_csv(os.path.join(data_path, 'data_characters.tsv'), sep='\t')

U = load_matrix('data_archetype_space_U.txt')
Sigma_vec = load_matrix('data_archetype_space_Sigma.txt')
V = load_matrix('data_archetype_space_V.txt')

# Ensure dimensions match U (464 traits)
k = U.shape[0]
Sigma_vec = Sigma_vec[:k]
V = V[:, :k]

Sigma = np.diag(Sigma_vec)

print(f"U shape: {U.shape}")
print(f"Sigma shape: {Sigma.shape}")
print(f"V shape: {V.shape}")
print(f"Traits: {len(traits_df)}, Characters: {len(characters_df)}")

Loading data...
U shape: (464, 464)
Sigma shape: (464,)
V shape: (2000, 464)
Traits: 464, Characters: 2000


## 1. Select Moral Traits

We need to find the indices of traits we consider "moral". 
Note: In this dataset, the $A$ matrix is typically constructed such that the `right pole` is positive and the `left pole` is negative. 
We will check the poles for each trait to ensure we assign the correct weight (e.g., +1 or -1) to represent the "Good" side.

In [7]:
# Search for moral traits
search_terms = ['kind', 'altruistic', 'honest', 'honorable', 'heroic', 'forgiving', 'virtuous']
matches = traits_df[traits_df['differential'].str.contains('|'.join(search_terms), case=False)]
matches[['index', 'differential', 'left pole', 'right pole']]

,index,differential,left pole,right pole
13,14,cunning :: honorable,cunning,honorable
24,25,forgiving :: vengeful,forgiving,vengeful
38,39,heroic :: villainous,heroic,villainous
78,79,selfish :: altruistic,selfish,altruistic
83,84,cruel :: kind,cruel,kind


In [8]:
# Define our Moral Traits and their 'Good' direction
# Format: {trait_index: direction}
# Direction is +1 if 'Good' is the right pole, -1 if 'Good' is the left pole.

moral_selection = {
    84: 1,   # cruel :: kind (Kind is right pole -> +1)
    79: 1,   # selfish :: altruistic (Altruistic is right pole -> +1)
    39: -1,  # heroic :: villainous (Heroic is left pole -> -1)
    14: 1    # cunning :: honorable (Honorable is right pole -> +1)
}

Am = np.zeros((U.shape[0], 1))
for idx, weight in moral_selection.items():
    # Subtract 1 because index in TSV is 1-based
    Am[idx-1] = weight

print(f"Moral vector Am created with {len(moral_selection)} traits.")

Moral vector Am created with 4 traits.


## 2. Project Morality into Archetype Space

The moral axis in the low-dimensional archetype space is given by $w_m = U^T A_m$.

In [9]:
wm = U.T @ Am
print(f"Moral axis wm shape: {wm.shape}")

Moral axis wm shape: (464, 1)


## 3. Project Characters onto the Moral Axis

Character coordinates in archetype space are $C = \Sigma V^T$.
The projection (score) of characters onto our moral axis is $S_{chars} = C^T w_m = (V \Sigma) w_m$.

In [10]:
character_scores = (V @ Sigma) @ wm
characters_df['moral_score'] = character_scores

print("Top 10 Moral Characters:")
display(characters_df.sort_values('moral_score', ascending=False).head(10)[['character', 'character/story', 'moral_score']])

print("\nTop 10 Immoral Characters:")
display(characters_df.sort_values('moral_score', ascending=True).head(10)[['character', 'character/story', 'moral_score']])

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 464 is different from 2000)

## 4. Find Morally-Aligned Traits

Which other traits (out of the 464) are most aligned with this moral axis?
The trait alignment scores are $S_{traits} = U \Sigma^2 w_m$.

In [ ]:
trait_scores = U @ (Sigma_vec**2 * wm.flatten())
traits_df['moral_alignment'] = trait_scores

print("Traits most aligned with 'Good':")
display(traits_df.sort_values('moral_alignment', ascending=False).head(15)[['differential', 'moral_alignment']])

print("\nTraits most aligned with 'Evil':")
display(traits_df.sort_values('moral_alignment', ascending=True).head(15)[['differential', 'moral_alignment']])

## 5. Moral Duality Exploration

Let's add a second axis (e.g., Masculine/Feminine) to create a 2D moral compass.
We can then see how characters cluster or spread.

In [ ]:
# Select a duality trait
# Index 4: masculine :: feminine
duality_idx = 4
Ad = np.zeros((U.shape[0], 1))
Ad[duality_idx-1] = 1 # Positive = Feminine

wd = U.T @ Ad
duality_scores = (V @ Sigma) @ wd
characters_df['duality_score'] = duality_scores

plt.figure(figsize=(12, 10))
sns.scatterplot(data=characters_df, x='moral_score', y='duality_score', alpha=0.5)

# Label some outliers
top_chars = characters_df.sort_values('moral_score', ascending=False).head(5)
bot_chars = characters_df.sort_values('moral_score', ascending=True).head(5)
extreme_duality = characters_df.sort_values('duality_score', ascending=False).head(5)
extreme_neg_duality = characters_df.sort_values('duality_score', ascending=True).head(5)

for i, row in pd.concat([top_chars, bot_chars, extreme_duality, extreme_neg_duality]).iterrows():
    plt.text(row['moral_score'], row['duality_score'], row['character'], fontsize=9)

plt.axvline(0, color='grey', linestyle='--')
plt.axhline(0, color='grey', linestyle='--')
plt.xlabel('Moral Score (Evil -> Good)')
plt.ylabel('Duality Score (Masculine -> Feminine)')
plt.title('2D Moral Compass: Morality vs Gender Duality')
plt.grid(True, alpha=0.3)
plt.show()

## 6. Clustering Moral "Types"

Finally, we can cluster characters based on their projection into a multi-dimensional "moral space" (using more moral traits or just the top components of morality).

In [ ]:
from sklearn.cluster import KMeans

# Let's use the moral score and a few other moral-aligned dimensions
moral_features = characters_df[['moral_score', 'duality_score']]

kmeans = KMeans(n_clusters=4, random_state=42)
characters_df['moral_cluster'] = kmeans.fit_predict(moral_features)

plt.figure(figsize=(10, 8))
sns.scatterplot(data=characters_df, x='moral_score', y='duality_score', hue='moral_cluster', palette='viridis', alpha=0.6)
plt.title('Character Clusters in Moral-Duality Space')
plt.show()